# LSTM Sequential Recommendation Model (Serverless GPU with PyTorch Lightning)

Deep learning model using PyTorch LSTM with distributed training on serverless GPU using **PyTorch Lightning**.

**Architecture:**
- Embedding layer (64 dims) for article representations
- 2-layer LSTM (128 hidden units) with dropout
- Fully connected layers (256 units)
- Output: Probability distribution over catalog

**Serverless GPU Configuration:**
- Uses `@distributed` decorator for A10 GPU acceleration
- PyTorch Lightning for automatic DDP setup
- Configurable number of GPUs (default: 8)
- Mixed precision training (FP16) for faster performance

**Key Benefits of PyTorch Lightning:**
- Automatic distributed data parallel (DDP) setup
- Built-in gradient synchronization
- MLflow integration for experiment tracking
- Early stopping and checkpointing
- Cleaner, more maintainable code

**Expected Performance:** MAP@12 ~ 0.015-0.025

## Setup

In [ ]:
# Install PyTorch Lightning for distributed training
%pip install lightning --quiet
dbutils.library.restartPython()

In [ ]:
import sys

# Add project root to path (go up 2 levels from notebooks/)
sys.path.append("../../")

from pyspark.sql.functions import *
import mlflow

from serverless_gpu import distributed

from config.catalog_config import *
from config.model_config import LSTM_CONFIG, EVAL_CONFIG
from config.widget_utils import get_widget_or_default, get_bundle_parameters
from data_engineering.data_utils import load_delta_table
from utils.evaluation_utils import log_evaluation_metrics
from utils.preprocessing_utils import (
    ArticleEncoder,
    create_customer_sequences,
    create_train_sequences,
)
from utils.pytorch_utils import LSTMRecommenderLightning, generate_recommendations

In [0]:
# Get parameters from bundle (passed as notebook parameters)
# Falls back to defaults when running interactively
params = get_bundle_parameters(model_default="lstm_model")
catalog_name = params["catalog_name"]
schema_name = params["schema_name"]
experiment_name = params["experiment_name"]
model_name = params["model_name"]

# GPU-specific parameters with defaults
num_gpus = int(get_widget_or_default("num_gpus", "8"))
gpu_type = get_widget_or_default("gpu_type", "A10")

print(f"Parameters:")
print(f"  Catalog: {catalog_name}")
print(f"  Schema: {schema_name}")
print(f"  Experiment: {experiment_name}")
print(f"  Model: {model_name}")
print(f"  Num GPUs: {num_gpus}")
print(f"  GPU Type: {gpu_type}")

## Configuration

In [0]:
# Model hyperparameters
config = LSTM_CONFIG.copy()
print("LSTM Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## Load and Prepare Data (On Driver)

In [0]:
print("Loading data from Delta tables...")

# Load transactions
transactions_df = load_delta_table(f"{catalog_name}.{schema_name}.transactions_bronze")

# Load train/val splits
train_df = load_delta_table(f"{catalog_name}.{schema_name}.train_transactions_silver")
val_df = load_delta_table(f"{catalog_name}.{schema_name}.val_transactions_silver")

# Load ground truth
val_ground_truth = load_delta_table(f"{catalog_name}.{schema_name}.val_ground_truth_silver")
test_ground_truth = load_delta_table(f"{catalog_name}.{schema_name}.test_ground_truth_silver")

print(f"Total transactions: {transactions_df.count():,}")
print(f"Train: {train_df.count():,}")
print(f"Val: {val_df.count():,}")
print(f"Val ground truth customers: {val_ground_truth.count():,}")

## Build Article Encoder

In [0]:
print("Building article vocabulary...")

# Get all unique articles from transactions
all_articles = (
    transactions_df
    .select("article_id")
    .distinct()
    .collect()
)
article_ids = [row["article_id"] for row in all_articles]

print(f"Total unique articles: {len(article_ids):,}")

# Create and fit encoder
encoder = ArticleEncoder()
encoder.fit(article_ids)

print(f"Vocabulary size (including padding): {encoder.vocab_size:,}")

## Create Purchase Sequences

In [0]:
print("Creating customer purchase sequences...")

# Get unique customers from each split
train_customers = train_df.select("customer_id").distinct()
val_customers = val_df.select("customer_id").distinct()

print(f"Train customers: {train_customers.count():,}")
print(f"Val customers: {val_customers.count():,}")

In [0]:
# Filter transactions by customer set
train_transactions = transactions_df.join(
    train_customers,
    "customer_id",
    "inner"
)

val_transactions = transactions_df.join(
    val_customers,
    "customer_id",
    "inner"
)

print(f"Train transactions: {train_transactions.count():,}")
print(f"Val transactions: {val_transactions.count():,}")

In [0]:
# Create sequences for each split
train_sequences = create_customer_sequences(
    train_transactions,
    max_sequence_length=config['sequence_length'] + 5,  # Extra for target
    min_sequence_length=config['min_sequence_length']
)

val_sequences = create_customer_sequences(
    val_transactions,
    max_sequence_length=config['sequence_length'] + 5,
    min_sequence_length=config['min_sequence_length']
)

print(f"Train sequences: {train_sequences.count():,}")
print(f"Val sequences: {val_sequences.count():,}")

# Show sample
print("\nSample sequence:")
sample = train_sequences.limit(1).collect()[0]
print(f"Customer: {sample['customer_id']}")
print(f"Sequence length: {sample['sequence_length']}")
print(f"First 5 articles: {sample['sequence'][:5]}")

## Create Training Pairs

In [ ]:
# Create input/target pairs for training
print("Creating train/val sequence pairs...")

train_pairs = create_train_sequences(
    train_sequences,
    sequence_length=config['sequence_length'],
    prediction_window=1  # Predict next 1 item
)

val_pairs = create_train_sequences(
    val_sequences,
    sequence_length=config['sequence_length'],
    prediction_window=1
)

print(f"Train pairs: {train_pairs.count():,}")
print(f"Val pairs: {val_pairs.count():,}")

# Show sample
print("\nSample training pair:")
sample = train_pairs.limit(1).collect()[0]
print(f"Customer: {sample['customer_id']}")
print(f"Input sequence: {sample['input_sequence'][:5]}... ({len(sample['input_sequence'])} items)")
print(f"Target articles: {sample['target_articles']}")

In [ ]:
# Convert to pandas for distributed training
import tempfile
import os
import pickle
import shutil
import uuid

print("Converting to pandas DataFrames...")
train_pairs_pd = train_pairs.toPandas()
val_pairs_pd = val_pairs.toPandas()

print(f"Train pairs (pandas): {len(train_pairs_pd):,} rows")
print(f"Val pairs (pandas): {len(val_pairs_pd):,} rows")

# Use Unity Catalog volume for shared storage accessible to distributed workers
# Volume path: /Volumes/{catalog}/{schema}/training_logs/
volume_base = f"/Volumes/{catalog_name}/{schema_name}/training_logs"

# Create unique directory for this training run
unique_id = str(uuid.uuid4())[:8]
temp_dir = f"{volume_base}/lstm_training_{unique_id}"

# Clean up if exists and recreate
if os.path.exists(temp_dir):
    print(f"Removing existing directory: {temp_dir}")
    shutil.rmtree(temp_dir)

os.makedirs(temp_dir, exist_ok=True)

train_data_path = f"{temp_dir}/train_pairs.pkl"
val_data_path = f"{temp_dir}/val_pairs.pkl"
encoder_path = f"{temp_dir}/encoder.pkl"

print(f"\nSaving data to Unity Catalog volume...")

# Use pickle to save data
with open(train_data_path, 'wb') as f:
    pickle.dump(train_pairs_pd, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(val_data_path, 'wb') as f:
    pickle.dump(val_pairs_pd, f, protocol=pickle.HIGHEST_PROTOCOL)

encoder.save(encoder_path)

print(f"✓ Train data saved to: {train_data_path}")
print(f"✓ Val data saved to: {val_data_path}")
print(f"✓ Encoder saved to: {encoder_path}")
print(f"\nNote: Using Unity Catalog volume for shared access across distributed GPU workers")

## Distributed Training Function

### Article Encoder

In [ ]:
import pickle
from typing import List


class ArticleEncoder:
    """Encode article IDs to integer indices"""

    def __init__(self):
        self.article_to_idx = {}
        self.idx_to_article = {}
        self.vocab_size = 0
        self.padding_idx = 0

    def fit(self, article_ids: List[str]):
        """Build vocabulary from article IDs"""
        unique_articles = sorted(set(article_ids))
        # Reserve index 0 for padding
        self.article_to_idx = {"<PAD>": 0}
        self.idx_to_article = {0: "<PAD>"}

        for idx, article_id in enumerate(unique_articles, start=1):
            self.article_to_idx[article_id] = idx
            self.idx_to_article[idx] = article_id

        self.vocab_size = len(self.article_to_idx)

    def encode(self, article_id: str) -> int:
        """Encode single article ID"""
        return self.article_to_idx.get(article_id, self.padding_idx)

    def encode_batch(self, article_ids: List[str]) -> List[int]:
        """Encode list of article IDs"""
        return [self.encode(aid) for aid in article_ids]

    def decode(self, idx: int) -> str:
        """Decode single index"""
        return self.idx_to_article.get(idx, "<PAD>")

    def decode_batch(self, indices: List[int]) -> List[str]:
        """Decode list of indices"""
        return [self.decode(idx) for idx in indices]

    def save(self, path: str):
        """Save encoder to pickle"""
        with open(path, "wb") as f:
            pickle.dump(
                {
                    "article_to_idx": self.article_to_idx,
                    "idx_to_article": self.idx_to_article,
                    "vocab_size": self.vocab_size,
                    "padding_idx": self.padding_idx,
                },
                f,
            )

    def load(self, path: str):
        """Load encoder from pickle"""
        with open(path, "rb") as f:
            data = pickle.load(f)
            self.article_to_idx = data["article_to_idx"]
            self.idx_to_article = data["idx_to_article"]
            self.vocab_size = data["vocab_size"]
            self.padding_idx = data["padding_idx"]

### Dataset

In [ ]:
import torch
from torch.utils.data import Dataset


class SequenceDataset(Dataset):
    """PyTorch Dataset for customer purchase sequences"""

    def __init__(
        self,
        customer_ids,
        input_sequences,
        target_articles,
        sequence_length,
        padding_idx=0,
    ):
        self.customer_ids = customer_ids
        self.input_sequences = input_sequences
        self.target_articles = target_articles
        self.sequence_length = sequence_length
        self.padding_idx = padding_idx

    def __len__(self):
        return len(self.customer_ids)

    def __getitem__(self, idx):
        input_seq = self.input_sequences[idx]
        target = self.target_articles[idx]
        customer_id = self.customer_ids[idx]

        # Pad input sequence
        if len(input_seq) < self.sequence_length:
            padded_seq = [self.padding_idx] * (
                self.sequence_length - len(input_seq)
            ) + input_seq
        else:
            padded_seq = input_seq[-self.sequence_length :]

        return (
            torch.tensor(padded_seq, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
            customer_id,
        )

### Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl


class LSTMRecommender(nn.Module):
    """LSTM-based recommendation model"""

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3,
        padding_idx=0,
    ):
        super(LSTMRecommender, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.padding_idx = padding_idx

        # Layers
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=padding_idx
        )
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.fc1 = nn.Linear(hidden_dim, 256)
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, vocab_size)

        # Initialize weights
        nn.init.xavier_uniform_(self.embedding.weight)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, input_seq, hidden=None):
        embedded = self.embedding(input_seq)
        lstm_out, hidden = self.lstm(embedded, hidden)
        last_output = lstm_out[:, -1, :]
        x = F.relu(self.fc1(last_output))
        x = self.dropout1(x)
        logits = self.fc2(x)
        return logits, hidden


class LSTMRecommenderLightning(pl.LightningModule):
    """PyTorch Lightning wrapper for LSTM model"""

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3,
        padding_idx=0,
        learning_rate=0.001,
        weight_decay=1e-5,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = LSTMRecommender(
            vocab_size, embedding_dim, hidden_dim, num_layers, dropout, padding_idx
        )
        self.criterion = nn.CrossEntropyLoss(ignore_index=padding_idx)

    def forward(self, input_seq, hidden=None):
        return self.model(input_seq, hidden)

    def training_step(self, batch, batch_idx):
        input_seq, target, customer_ids = batch
        logits, _ = self.model(input_seq)
        target_first = target[:, 0]
        loss = self.criterion(logits, target_first)
        _, predicted = torch.max(logits, 1)
        accuracy = (predicted == target_first).float().mean()
        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        self.log(
            "train_accuracy",
            accuracy,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        return loss

    def validation_step(self, batch, batch_idx):
        input_seq, target, customer_ids = batch
        logits, _ = self.model(input_seq)
        target_first = target[:, 0]
        loss = self.criterion(logits, target_first)
        _, predicted = torch.max(logits, 1)
        accuracy = (predicted == target_first).float().mean()
        self.log(
            "val_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        self.log(
            "val_accuracy",
            accuracy,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay,
        )

In [ ]:
# Define the distributed training function using PyTorch Lightning
# All code is inlined to avoid module import issues on GPU workers
@distributed(gpus=num_gpus, gpu_type=gpu_type, remote=True)
def train_lstm_lightning(
    train_data_path,
    val_data_path,
    encoder_path,
    config_dict,
    experiment_path,
    catalog,
    schema,
    model_name_str,
):
    """
    Distributed training function for LSTM model using PyTorch Lightning.
    All dependencies are inlined to avoid import issues on GPU workers.
    """
    import pickle
    import mlflow
    import mlflow.pytorch
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import MLFlowLogger
    from torch.utils.data import DataLoader

    # ========== MAIN TRAINING CODE ==========
    print(f"Loading training data from {train_data_path}...")

    with open(train_data_path, "rb") as f:
        train_data_pd = pickle.load(f)
    with open(val_data_path, "rb") as f:
        val_data_pd = pickle.load(f)

    print(f"✓ Train data loaded: {len(train_data_pd):,} rows")
    print(f"✓ Val data loaded: {len(val_data_pd):,} rows")

    # Load encoder
    encoder_obj = ArticleEncoder()
    encoder_obj.load(encoder_path)
    print(f"✓ Encoder loaded: vocab_size={encoder_obj.vocab_size:,}")

    # Prepare datasets
    def pd_to_dataset(df_pd):
        customer_ids = []
        input_sequences = []
        target_articles = []
        for _, row in df_pd.iterrows():
            customer_ids.append(row["customer_id"])
            input_seq_encoded = encoder_obj.encode_batch(row["input_sequence"])
            target_encoded = encoder_obj.encode_batch(row["target_articles"])
            input_sequences.append(input_seq_encoded)
            target_articles.append(target_encoded)
        return SequenceDataset(
            customer_ids,
            input_sequences,
            target_articles,
            config_dict["sequence_length"],
            encoder_obj.padding_idx,
        )

    train_dataset = pd_to_dataset(train_data_pd)
    val_dataset = pd_to_dataset(val_data_pd)

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config_dict["batch_size"],
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config_dict["batch_size"],
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")

    # Initialize model
    model = LSTMRecommenderLightning(
        vocab_size=encoder_obj.vocab_size,
        embedding_dim=config_dict["embedding_dim"],
        hidden_dim=config_dict["hidden_dim"],
        num_layers=config_dict["num_layers"],
        dropout=config_dict["dropout"],
        padding_idx=encoder_obj.padding_idx,
        learning_rate=config_dict["learning_rate"],
        weight_decay=config_dict["weight_decay"],
    )

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    # Setup MLflow
    mlflow.set_experiment(experiment_path)
    mlflow_logger = MLFlowLogger(
        experiment_name=experiment_path, run_name="lstm_lightning_a10"
    )

    # Callbacks
    early_stop_callback = EarlyStopping(
        monitor="val_loss",
        patience=config_dict["early_stopping_patience"],
        mode="min",
        verbose=True,
    )
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="lstm-{epoch:02d}-{val_loss:.4f}",
    )

    # Initialize trainer
    trainer = pl.Trainer(
        max_epochs=config_dict["num_epochs"],
        accelerator="gpu",
        logger=mlflow_logger,
        callbacks=[early_stop_callback, checkpoint_callback],
        gradient_clip_val=5.0,
        log_every_n_steps=50,
        enable_progress_bar=True,
        precision="16-mixed",
    )

    # Log hyperparameters
    mlflow_logger.log_hyperparams(
        {
            "model_type": "lstm_lightning",
            "vocab_size": encoder_obj.vocab_size,
            "embedding_dim": config_dict["embedding_dim"],
            "hidden_dim": config_dict["hidden_dim"],
            "num_layers": config_dict["num_layers"],
            "dropout": config_dict["dropout"],
            "batch_size": config_dict["batch_size"],
            "learning_rate": config_dict["learning_rate"],
            "sequence_length": config_dict["sequence_length"],
            "num_epochs": config_dict["num_epochs"],
            "total_parameters": total_params,
        }
    )

    # Train
    print("\n" + "=" * 60)
    print("Starting training with PyTorch Lightning...")
    print("=" * 60)
    trainer.fit(model, train_loader, val_loader)
    print("\n" + "=" * 60)
    print("Training completed!")
    print("=" * 60)

    # Load best model
    best_model_path = checkpoint_callback.best_model_path
    print(f"Best model checkpoint: {best_model_path}")
    best_model = LSTMRecommenderLightning.load_from_checkpoint(best_model_path)

    # Log model to MLflow
    with mlflow.start_run(run_id=mlflow_logger.run_id):
        mlflow.log_artifact(encoder_path, "encoder")
        mlflow.pytorch.log_model(best_model.model, "model")
        mlflow.set_tag("stage", "training")
        mlflow.set_tag("model_type", "lstm")
        mlflow.set_tag("framework", "pytorch_lightning")
        mlflow.set_tag("distributed", "true")

    run_id = mlflow_logger.run_id
    print(f"\nMLflow Run ID: {run_id}")

    # Extract metrics
    metrics = trainer.callback_metrics
    history = {
        "best_val_loss": float(metrics.get("val_loss", 0)),
        "best_val_accuracy": float(metrics.get("val_accuracy", 0)),
    }

    return {"run_id": run_id, "history": history, "total_params": total_params}

## Run Distributed Training

In [ ]:
print("Starting distributed training with PyTorch Lightning on serverless GPU...")

# Initialize MLflow with experiment from parameters
mlflow.set_experiment(experiment_name)

# Run distributed training with Lightning
result = train_lstm_lightning.distributed(
    train_data_path=train_data_path,
    val_data_path=val_data_path,
    encoder_path=encoder_path,
    config_dict=config,
    experiment_path=experiment_name,
    catalog=catalog_name,
    schema=schema_name,
    model_name_str=model_name
)

# Extract results
run_id = result['run_id']
history = result['history']
total_params = result['total_params']

print(f"\n✓ Distributed training with PyTorch Lightning complete!")
print(f"✓ MLflow Run ID: {run_id}")
print(f"✓ Best Val Loss: {history['best_val_loss']:.4f}")
print(f"✓ Best Val Accuracy: {history['best_val_accuracy']:.4f}")

## Load Trained Model for Inference

In [ ]:
print("Loading trained model for inference...")

# Load model from MLflow run
model_uri = f"runs:/{run_id}/model"
loaded_model = mlflow.pytorch.load_model(model_uri)

# Set to evaluation mode
loaded_model.eval()

# Move to GPU if available (single GPU for inference)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model = loaded_model.to(device)

print(f"✓ Model loaded on device: {device}")

## Generate Recommendations for Validation Set

In [0]:
print("Generating recommendations for validation customers...")

# Get all validation customers
val_customer_ids_list = [row['customer_id'] for row in val_ground_truth.select('customer_id').collect()]

print(f"Validation customers: {len(val_customer_ids_list):,}")

In [0]:
# Get sequences for validation customers
val_sequences_dict = {}
for row in val_sequences.collect():
    val_sequences_dict[row['customer_id']] = row['sequence']

# Prepare test input sequences
test_input_sequences = []
test_customer_ids = []

for customer_id in val_customer_ids_list:
    if customer_id in val_sequences_dict:
        sequence = val_sequences_dict[customer_id]

        # Encode sequence
        encoded_seq = encoder.encode_batch(sequence[-config['sequence_length']:])

        # Pad if necessary
        if len(encoded_seq) < config['sequence_length']:
            padded = [encoder.padding_idx] * (config['sequence_length'] - len(encoded_seq)) + encoded_seq
        else:
            padded = encoded_seq[-config['sequence_length']:]

        test_customer_ids.append(customer_id)
        test_input_sequences.append(torch.tensor(padded, dtype=torch.long))

print(f"Generating recommendations for {len(test_customer_ids):,} customers with sufficient history")

In [0]:
# Generate recommendations
recommendations = generate_recommendations(
    model=loaded_model,
    input_sequences=test_input_sequences,
    customer_ids=test_customer_ids,
    encoder=encoder,
    device=device,
    k=12,
    batch_size=config['batch_size']
)

print(f"Generated recommendations for {len(recommendations)} customers")

# Show sample
if len(test_customer_ids) > 0:
    sample_customer = test_customer_ids[0]
    print(f"\nSample recommendations for customer {sample_customer}:")
    print(recommendations[sample_customer])

## Convert to Spark DataFrame

In [0]:
print("Converting recommendations to DataFrame...")

# Convert to list of tuples
recommendations_list = [
    (customer_id, article_ids)
    for customer_id, article_ids in recommendations.items()
]

# Create DataFrame
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

predictions_df = spark.createDataFrame(
    recommendations_list,
    schema=["customer_id", "predicted_articles"]
)

print(f"Predictions DataFrame: {predictions_df.count()} rows")
display(predictions_df.limit(5))

## Evaluate on Validation Set

In [0]:
print("Evaluating model on validation set...")

# Evaluate (will log to MLflow automatically)
with mlflow.start_run(run_id=run_id):
    metrics = log_evaluation_metrics(
        predictions_df,
        val_ground_truth,
        "LSTM Distributed (A10)",
        k=12
    )

print(f"\n{'='*60}")
print(f"LSTM Model Performance (Distributed A10)")
print(f"{'='*60}")
print(f"MAP@12: {metrics['map@12']:.6f}")
if 'map@5' in metrics:
    print(f"MAP@5: {metrics['map@5']:.6f}")
if 'catalog_coverage' in metrics:
    print(f"Coverage: {metrics['catalog_coverage']:.4f}")
print(f"Customers Evaluated: {metrics['num_customers']:,}")
print(f"{'='*60}")

## Register Model to Unity Catalog

In [0]:
print("Registering model to Unity Catalog Model Registry...")

# Register the PyTorch model to Unity Catalog
uc_model_name = f"{catalog_name}.{schema_name}.{model_name}"
model_uri = f"runs:/{run_id}/model"

print(f"\nRegistering model to Unity Catalog: {uc_model_name}")

registered_model = mlflow.register_model(model_uri, uc_model_name)

print(f"✓ Model registered: {uc_model_name}")
print(f"✓ Version: {registered_model.version}")
print(f"✓ Run ID: {run_id}")

# Add model description
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.update_registered_model(
    name=uc_model_name,
    description=f"PyTorch LSTM sequential recommendation model (Distributed A10 GPU). "
    f"MAP@12: {metrics['map@12']:.6f}. "
    f"Trained on {num_gpus}x {gpu_type} GPUs with {total_params:,} parameters.",
)

client.update_model_version(
    name=uc_model_name,
    version=registered_model.version,
    description=f"LSTM architecture: Embedding({config['embedding_dim']}) → "
    f"LSTM({config['num_layers']} layers, {config['hidden_dim']} units) → Dense(256). "
    f"MAP@12: {metrics['map@12']:.6f}. "
    f"Vocabulary: {encoder.vocab_size:,} articles, Sequence length: {config['sequence_length']}. "
    f"Trained with distributed training on {num_gpus}x {gpu_type} GPUs.",
)

print(f"\n✓ Model {uc_model_name} v{registered_model.version} registered successfully!")

## Save Predictions

In [0]:
# Save predictions to gold schema
output_table = f"{catalog_name}.{schema_name}.lstm_predictions_gold"
print(f"Saving predictions to {output_table}...")

predictions_df \
    .withColumn("model_type", lit("lstm_distributed_a10")) \
    .withColumn("run_id", lit(run_id)) \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✓ Predictions saved to {output_table}")

## Training Visualization

In [ ]:
# Training metrics visualization
# Note: PyTorch Lightning logs metrics to MLflow automatically
# You can view detailed training curves in the MLflow UI

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Display best validation metrics
metrics_text = f"""
Training Complete - Best Model Metrics

Best Validation Loss: {history['best_val_loss']:.4f}
Best Validation Accuracy: {history['best_val_accuracy']:.4f}

Model Configuration:
- Vocabulary Size: {encoder.vocab_size:,}
- Embedding Dim: {config['embedding_dim']}
- Hidden Dim: {config['hidden_dim']}
- Num Layers: {config['num_layers']}
- Total Parameters: {total_params:,}
- GPUs Used: {num_gpus}x {gpu_type}

View detailed training curves in MLflow UI:
Run ID: {run_id}
"""

ax.text(0.5, 0.5, metrics_text, 
        horizontalalignment='center',
        verticalalignment='center',
        transform=ax.transAxes,
        fontsize=12,
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.axis('off')

plt.tight_layout()
display(fig)

print("\n" + "="*60)
print("For detailed training curves, view the MLflow experiment:")
print(f"Experiment: {experiment_name}")
print(f"Run ID: {run_id}")
print("="*60)

## Summary

In [0]:
print("\n" + "="*60)
print("LSTM MODEL TRAINING COMPLETE (SERVERLESS GPU)")
print("="*60)
print(f"Model Type: Sequential LSTM (PyTorch + Distributed)")
print(f"GPU Configuration: {num_gpus}x {gpu_type} (Serverless)")
print(f"")
print(f"Architecture:")
print(f"  - Embedding: {config['embedding_dim']} dims")
print(f"  - LSTM: {config['num_layers']} layers x {config['hidden_dim']} units")
print(f"  - Total Parameters: {total_params:,}")
print(f"")
print(f"Data:")
print(f"  - Vocabulary Size: {encoder.vocab_size:,} articles")
print(f"  - Sequence Length: {config['sequence_length']}")
print(f"  - Training Samples: {len(train_pairs_pd):,}")
print(f"  - Validation Samples: {len(recommendations):,}")
print(f"")
print(f"Performance:")
print(f"  - MAP@12: {metrics['map@12']:.6f}")
if 'map@5' in metrics:
    print(f"  - MAP@5: {metrics['map@5']:.6f}")
if 'catalog_coverage' in metrics:
    print(f"  - Coverage: {metrics['catalog_coverage']:.4f}")
print(f"")
print(f"MLflow:")
print(f"  - Experiment: {experiment_name}")
print(f"  - Run ID: {run_id}")
print(f"  - Model: {uc_model_name}")
print(f"  - Predictions: {output_table}")
print("="*60)
print("\n✓ LSTM training with serverless GPU (A10) complete!")
print("="*60)